# Importing Required Packages

In [1]:
# Required packages
import pandas as pd  # Load and preprocess data.
import os  # Manage file paths and directories.
import requests  # Send HTTP requests to interact with web APIs or download content from the internet.
import io  # Provides tools for working with I/O streams.
import matplotlib.pyplot as plt  # Create static visualizations like line plots, bar charts, and scatter plots.
import seaborn as sns  # Create enhanced statistical data visualizations like heatmaps and pair plots.
import plotly.express as px  # Quickly create interactive visualizations like scatter plots, line charts, and maps.
import plotly.graph_objects as go  # Build detailed and customized interactive visualizations.
import numpy as np  # Perform numerical operations on arrays and matrices efficiently.
import math  # Perform basic mathematical calculations like logarithms and trigonometric functions.
import ee  # Use Google Earth Engine for large-scale geospatial analysis and satellite data processing.
import scipy.stats as stats  # Perform statistical functions and hypothesis testing.
from sklearn.preprocessing import StandardScaler  # Standardize features by removing the mean and scaling to unit variance.
from sklearn.model_selection import train_test_split  # Split data into training and testing sets for model evaluation.
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor  # Use ensemble methods for regression tasks.
from sklearn.neural_network import MLPRegressor  # Use a Multi-Layer Perceptron model for regression.
from sklearn.svm import SVR  # Use a Support Vector Regressor for regression problems.
from sklearn.metrics import mean_squared_error, r2_score  # Evaluate regression models using metrics like MSE and R².
from sklearn.model_selection import GridSearchCV
from xgboost.sklearn import XGBRegressor
import json

In [14]:
url = "https://raw.githubusercontent.com/irungus/agf_kenya/main/data/acacia_df_cleaned_agb_14_2_2025.csv"
acacia_df_cleaned_agb = pd.read_csv(url)

In [15]:
cols_to_drop = [
    'FID','start','end','Enumerator','agf','plotID','transect','County',
    'localname','genus','species','newspecies','height','canopyd','latitude',
    'longitude','altitude','accuracy','__version_','_version_','key','KEY',
    'date','genusCorr','speciesCorr','sdWD','family'
]

acacia_df_cleaned_agb = acacia_df_cleaned_agb.drop(columns=cols_to_drop, errors='ignore')
acacia_df_cleaned_agb = acacia_df_cleaned_agb[['dbh', 'height_m', 'AGB', 'genus_species','meanWD']]

In [16]:
acacia_df_cleaned_agb.head()

,dbh,height_m,AGB,genus_species,meanWD
0,20,1.8,39.236401,Acacia mellifera,0.947000
1,36,13.0,711.287343,Acacia tortilis,0.787831
2,48,7.3,653.304165,Acacia nilotica,0.723333
3,13,4.5,31.818687,Acacia nilotica,0.723333
4,6,1.8,2.876250,Acacia nilotica,0.723333


In [17]:
acacia_df_cleaned_agb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   dbh            850 non-null    int64  
 1   height_m       850 non-null    float64
 2   AGB            850 non-null    float64
 3   genus_species  850 non-null    object 
 4   meanWD         850 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 33.3+ KB


In [18]:
acacia_df_cleaned_agb.describe()

,dbh,height_m,AGB,meanWD
count,850.000000,850.000000,850.000000,850.000000
mean,42.902353,5.288094,974.338306,0.772008
std,35.663968,3.513183,2327.387025,0.065448
min,5.000000,0.800000,1.113433,0.659667
25%,18.250000,2.500000,51.913377,0.723333
50%,32.000000,4.500000,204.767951,0.787831
75%,55.750000,7.000000,714.844172,0.787831
max,200.000000,22.000000,23249.279980,0.947000


In [23]:
def calculate_stats_by_genus_species(dataframe):
    """
    Calculates the mean, standard deviation, and count of height, dbh, wood density as calcualated
    from Biomass package and AGB the for each unique genus-species combination.
    """
    required_columns = {'genus_species', 'height_m', 'dbh'}
    if not required_columns.issubset(dataframe.columns):
        raise ValueError(f"DataFrame must contain columns: {required_columns}")
    # Group by genus and species
    grouped = dataframe.groupby(['genus_species'])
    # Calculate statistics for height_m and dbh
    stats = grouped.agg(
        count=('genus_species', 'count'),
        height_mean=('height_m', 'mean'),
        height_std=('height_m', 'std'),
        height_min=('height_m', 'min'),
        height_max=('height_m', 'max'),
        dbh_mean=('dbh', 'mean'),
        dbh_std=('dbh', 'std'),
        dbh_min=('dbh', 'min'),
        dbh_max=('dbh', 'max'),
        WoodDensity_mean = ('meanWD', 'mean'),
        AGB_mean=('AGB', 'mean'),
        AGB_std=('AGB', 'std'),
        AGB_min=('AGB', 'min'),
        AGB_max=('AGB', 'max')
    ).reset_index()
    return stats

In [24]:
stats_summary_acacia = calculate_stats_by_genus_species(acacia_df_cleaned_agb)
stats_summary_acacia

,genus_species,count,height_mean,height_std,height_min,height_max,dbh_mean,dbh_std,dbh_min,dbh_max,WoodDensity_mean,AGB_mean,AGB_std,AGB_min,AGB_max
0,Acacia drepanolobium,138,3.445290,1.842462,0.8,8.5,25.768116,21.786385,5,120,0.787831,184.714302,353.262078,1.113433,2943.514584
1,Acacia mearnsii,103,7.309709,3.190166,1.0,20.0,46.466019,35.494771,5,190,0.659667,983.103737,1574.388882,2.041129,9576.761484
2,Acacia mellifera,63,3.102540,1.920459,0.9,8.0,35.476190,37.120057,5,200,0.947000,351.195782,815.738057,2.336378,5783.803635
3,Acacia nilotica,114,4.044737,2.953065,1.0,19.0,28.885965,19.475766,5,110,0.723333,321.260992,885.823825,1.135322,7955.617717
4,Acacia tortilis,334,5.285240,3.173127,1.0,20.0,49.071856,35.337764,5,200,0.787831,1170.559391,2599.983199,1.234023,23249.279980
5,Acacia xanthophloea,98,8.619388,4.488597,1.0,22.0,63.336735,47.718017,5,200,0.758000,2568.585487,3967.427724,3.472418,19471.144160
